# 03 · Car-park records in the canonical pool

**Issue:** obvious parking-space *canonical* records leak into the candidate pool and
sometimes **win** against real dwellings.

```
dwelling : FLAT 5 BASEMENT 447 EXAMPLE ROAD LONDON
parking  : CAR PARK SPACE 5 EXAMPLE COURT 447 EXAMPLE ROAD LONDON
```

A text check for `CAR PARK SPACE` / `CAR PARKING SPACE` reportedly matches roughly
**16,659** canonical rows nationally (52 in Hackney). The proposed first step:
**exclude obvious parking-space canonical records during matching** with a *narrow*
text rule, leaving broader garage filtering for later.

This notebook reproduces the failure conceptually on tiny data, shows the fix, and
checks the rule is safe (doesn't eat real "garage" dwellings). Runs offline.

## Why a parking record competes with a dwelling

Matching strength comes from **shared, rare tokens**. The parking and dwelling records
above share the building number `447`, the street `EXAMPLE ROAD`, the town `LONDON`
*and the same postcode* — so they land in the same candidate pool (blocking) and share
most of the high-value tokens. The only tokens that distinguish them (`CAR`, `PARK`,
`SPACE`, `BASEMENT`) are a small part of the signal. When the messy address is itself
sparse or ambiguous, the parking record can edge out the real dwelling.

In [1]:
import duckdb
from uk_address_matcher import AddressMatcher, SplinkStage

# Canonical pool for one building (same postcode) — a dwelling, a ground flat,
# a CAR PARK SPACE record, plus the user's "garage" examples to test safety later.
CANON_SQL = """
SELECT * FROM (VALUES
  ('c_flat5b',  'FLAT 5 BASEMENT 447 EXAMPLE ROAD LONDON',                'E1 6AA'),
  ('c_gflat5',  'GROUND FLAT 5 447 EXAMPLE ROAD LONDON',                  'E1 6AA'),
  ('c_carpark', 'CAR PARK SPACE 5 EXAMPLE COURT 447 EXAMPLE ROAD LONDON', 'E1 6AA'),
  ('c_garage1', 'FLAT OVER GARAGE 8 EXAMPLE ROAD LONDON',                 'E1 6AA'),
  ('c_garage2', 'FIRST FLAT 1 EXAMPLE GARAGE YARD SUMMER LANE LONDON',    'E1 6AA')
) t(unique_id, address_concat, postcode)
"""

def run_match(messy_sql, canonical_filter=None):
    """Fresh connection per run: Splink registers temp tables that clash on reuse."""
    con = duckdb.connect(":memory:")
    matcher = AddressMatcher(
        canonical_addresses=con.sql(CANON_SQL),
        addresses_to_match=con.sql(messy_sql),
        con=con,
        canonical_address_filter=canonical_filter,
        stages=[SplinkStage(
            predict_threshold_match_weight=-30,
            final_match_weight_threshold=-30,   # keep everything so we can SEE the candidates
            include_full_postcode_block=True,
        )],
    )
    return matcher

con0 = duckdb.connect(":memory:")
con0.sql(CANON_SQL).df()

,unique_id,address_concat,postcode
0,c_flat5b,FLAT 5 BASEMENT 447 EXAMPLE ROAD LONDON,E1 6AA
1,c_gflat5,GROUND FLAT 5 447 EXAMPLE ROAD LONDON,E1 6AA
2,c_carpark,CAR PARK SPACE 5 EXAMPLE COURT 447 EXAMPLE ROA...,E1 6AA
3,c_garage1,FLAT OVER GARAGE 8 EXAMPLE ROAD LONDON,E1 6AA
4,c_garage2,FIRST FLAT 1 EXAMPLE GARAGE YARD SUMMER LANE L...,E1 6AA


## The parking record is in the candidate pool

For a messy `FLAT 5 447 EXAMPLE ROAD LONDON`, list every candidate Splink scored.
The car-park record shows up as a genuine candidate.

In [2]:
m = run_match("SELECT * FROM (VALUES ('m_1','FLAT 5 447 EXAMPLE ROAD LONDON','E1 6AA')) t(unique_id,address_concat,postcode)")
res = m.match()
sp = res._splink_predictions(limit=50)
sp.select(
    "round(match_weight,2) AS match_weight, unique_id_l AS canonical_id, "
    "original_address_concat_l AS canonical_address"
).order("match_weight DESC").df()

,match_weight,canonical_id,canonical_address
0,19.55,c_gflat5,GROUND FLAT 5 447 EXAMPLE ROAD LONDON
1,10.98,c_flat5b,FLAT 5 BASEMENT 447 EXAMPLE ROAD LONDON
2,-0.02,c_carpark,CAR PARK SPACE 5 EXAMPLE COURT 447 EXAMPLE ROA...


## The failure mode: when the parking record **wins**

Make the messy address ambiguous — closer to the parking record's distinctive tokens
(`SPACE`, `EXAMPLE COURT`). Now the car-park record is the top match.

In [3]:
AMBIGUOUS = ("SELECT * FROM (VALUES "
             "('m_amb','SPACE 5 EXAMPLE COURT 447 EXAMPLE ROAD LONDON','E1 6AA')) "
             "t(unique_id,address_concat,postcode)")

without_filter = run_match(AMBIGUOUS).match().matches()
without_filter.select(
    "unique_id, resolved_canonical_id, original_address_concat_canonical, round(match_weight,2) AS mw"
).df()

,unique_id,resolved_canonical_id,original_address_concat_canonical,mw
0,m_amb,c_carpark,CAR PARK SPACE 5 EXAMPLE COURT 447 EXAMPLE ROA...,37.3


## The fix — `canonical_address_filter`

`AddressMatcher` accepts `canonical_address_filter`: a DuckDB SQL boolean expression
applied to the canonical input **before** matching. We keep the rule **narrow** — only
`CAR PARK SPACE` / `CAR PARKING SPACE`:

```python
PARKING_FILTER = "NOT regexp_matches(upper(address_concat), 'CAR PARK(ING)? SPACE')"
```

> Note on the column name: for a **raw** canonical relation the column is
> `address_concat` (as here). For a **prepared canonical folder** the filter runs against
> the prepared columns — use `original_address_concat` there.

In [4]:
PARKING_FILTER = "NOT regexp_matches(upper(address_concat), 'CAR PARK(ING)? SPACE')"

with_filter = run_match(AMBIGUOUS, canonical_filter=PARKING_FILTER).match().matches()
with_filter.select(
    "unique_id, resolved_canonical_id, original_address_concat_canonical, round(match_weight,2) AS mw"
).df()

,unique_id,resolved_canonical_id,original_address_concat_canonical,mw
0,m_amb,c_flat5b,FLAT 5 BASEMENT 447 EXAMPLE ROAD LONDON,4.28


The winner flips from the car-park record to a real dwelling once parking records are
excluded from the pool. That is the whole point of the issue's "first step".

## Is the narrow rule safe? Check it doesn't eat real dwellings

The user flagged genuinely-residential addresses that merely contain the word
*garage*: `FLAT OVER GARAGE ...`, `... EXAMPLE GARAGE YARD ...`. A *broad* garage filter
would wrongly drop these. Our narrow `CAR PARK(ING)? SPACE` rule leaves them untouched —
let's confirm by listing which canonical rows the filter removes vs keeps.

In [5]:
con = duckdb.connect(":memory:")
canon = con.sql(CANON_SQL)
flagged = canon.select(
    "unique_id, address_concat, "
    "regexp_matches(upper(address_concat), 'CAR PARK(ING)? SPACE') AS removed_by_filter"
)
flagged.df()

,unique_id,address_concat,removed_by_filter
0,c_flat5b,FLAT 5 BASEMENT 447 EXAMPLE ROAD LONDON,False
1,c_gflat5,GROUND FLAT 5 447 EXAMPLE ROAD LONDON,False
2,c_carpark,CAR PARK SPACE 5 EXAMPLE COURT 447 EXAMPLE ROA...,True
3,c_garage1,FLAT OVER GARAGE 8 EXAMPLE ROAD LONDON,False
4,c_garage2,FIRST FLAT 1 EXAMPLE GARAGE YARD SUMMER LANE L...,False


In [6]:
removed = canon.filter("regexp_matches(upper(address_concat), 'CAR PARK(ING)? SPACE')")
kept    = canon.filter("NOT regexp_matches(upper(address_concat), 'CAR PARK(ING)? SPACE')")
print("removed:", removed.aggregate("count(*) n").fetchone()[0],
      "| kept:", kept.aggregate("count(*) n").fetchone()[0])
print("removed ids:", [r[0] for r in removed.select('unique_id').fetchall()])

removed: 1 | kept: 4
removed ids: ['c_carpark']


## Scoping & where this belongs

- **Narrow first (recommended).** `CAR PARK SPACE` / `CAR PARKING SPACE` is high-precision:
  these are unambiguous non-dwellings. Broader terms (`GARAGE`, `PARKING`) risk false
  positives like *FLAT OVER GARAGE* — defer until proven safe with data.
- **Where to apply.** Today the cleanest lever is `canonical_address_filter` at
  `AddressMatcher` construction (raw relation) **or** when calling
  `prepare_canonical_folder` / `load_prepared_canonical_data` (prepared folders).
- **Longer term.** The issue frames this as a *raw canonical-data* hygiene problem.
  A reusable, well-tested `canonical` cleaning step (or a documented, shipped default
  filter expression) would beat every caller re-inventing the regex. That is the
  natural shape of the contribution: ship the narrow rule + tests, document the column
  caveat (raw vs prepared), and leave broader garage filtering as a follow-up.

See `scripts/parking_filter.py` for a reusable version of this filter.